In [1]:
# Quantitative Analysis for DJI M350 RTK Performance
# Parameters derived from image_ef8f60.png and aerodynamic principles

def calculate_mission_impact(battery_wh, hover_p, lift_gain, speed_ms, sensor_p):
    """
    Calculates drone performance based on raw electrical and aerodynamic inputs.
    """
    # 1. Aerodynamic Power Model
    # Induced Power: Power to stay airborne, reduced by translational lift gain
    p_induced_moving = hover_p * (1 - lift_gain)
    
    # Parasitic Power: Calculated based on the cube of speed
    # Reference: 150W is the verified drag for this drone class at 12 m/s
    p_parasitic = 150.0 * (speed_ms / 12.0)**3
    
    # Total System Power Draw
    total_p = p_induced_moving + p_parasitic + sensor_p
    
    # 2. Performance Metrics
    flight_time_mins = (battery_wh / total_p) * 60
    distance_km = (speed_ms * 3.6) * (flight_time_mins / 60)
    
    # Area Mapped (Assuming 150m swath width for wildfire thermal mapping)
    area_km2 = distance_km * 0.150 
    
    return {
        "Total Power (W)": round(total_p, 2),
        "Flight Time (min)": round(flight_time_mins, 2),
        "Distance (km)": round(distance_km, 2),
        "Area (km2)": round(area_km2, 3)
    }



# Performance km and flight time

## Standard drone

In [2]:
# --- DEFINE PARAMETERS HERE ---
# Battery: 526.4 Wh (From two TB65 batteries in image_ef8f60.png)
# Hover Power: ~717 W for M350 RTK class
# Lift Gain: 0.20 (20% efficiency increase when moving)
# Sensor Power: 28.0 (Standard) vs 0.01 (Neuromorphic/SNN)

results = calculate_mission_impact(
    battery_wh = 526.4, 
    hover_p = 717.0, 
    lift_gain = 0.20, 
    speed_ms = 12.0, 
    sensor_p = 28
)

print(f"Results: {results}")

Results: {'Total Power (W)': 751.6, 'Flight Time (min)': 42.02, 'Distance (km)': 30.26, 'Area (km2)': 4.538}


## Optimzed with SNN

### Standard flight speed

In [3]:
# --- DEFINE PARAMETERS HERE ---
# Battery: 526.4 Wh (From two TB65 batteries in image_ef8f60.png)
# Hover Power: ~717 W for M350 RTK class
# Lift Gain: 0.20 (20% efficiency increase when moving)
# Sensor Power: 28.0 (Standard) vs 0.01 (Neuromorphic/SNN)

results = calculate_mission_impact(
    battery_wh = 526.4, 
    hover_p = 717.0, 
    lift_gain = 0.20, 
    speed_ms = 12.0, 
    sensor_p = 0.01
)

print(f"Results: {results}")

Results: {'Total Power (W)': 723.61, 'Flight Time (min)': 43.65, 'Distance (km)': 31.43, 'Area (km2)': 4.714}


### Max flight speed

In [4]:
# --- DEFINE PARAMETERS HERE ---
# Battery: 526.4 Wh (From two TB65 batteries in image_ef8f60.png)
# Hover Power: ~717 W for M350 RTK class
# Lift Gain: 0.20 (20% efficiency increase when moving)
# Sensor Power: 28.0 (Standard) vs 0.01 (Neuromorphic/SNN)

results = calculate_mission_impact(
    battery_wh = 526.4, 
    hover_p = 717.0, 
    lift_gain = 0.20, 
    speed_ms = 20.0, 
    sensor_p = 0.01
)

print(f"Results: {results}")

Results: {'Total Power (W)': 1268.05, 'Flight Time (min)': 24.91, 'Distance (km)': 29.89, 'Area (km2)': 4.483}


## Vizualizing

In [18]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

def run_comprehensive_mission(
    battery_wh=526.4, 
    hover_p=717.0, 
    lift_gain=0.20, 
    speed_ms=18.0, 
    sensor_p=0.01,
    fire_growth_rate=0.15, 
    real_to_sim_ratio=300, # 1s GIF = 5 min real-world
    save_path="large_map_mission.gif"
):
    # 1. Physical Constants & Scaled Map Size
    MAP_SIZE = 3000  # Increased map boundary
    BASE_POS = np.array([300.0, 300.0])
    FIRE_CENTER = np.array([1500.0, 1500.0]) # Centered in the 3000m grid
    
    # Power logic derived from DJI TB65 and Aerodynamic Principles
    p_parasitic = 150.0 * (speed_ms / 12.0)**3
    total_power_w = (hover_p * (1 - lift_gain)) + p_parasitic + sensor_p
    battery_joules = battery_wh * 3600
    max_battery = battery_joules
    
    # 2. Time Scaling Logic (Adjusted for 30 FPS)
    fps = 30 # User requested frame rate
    dt_real = real_to_sim_ratio / fps 
    total_flight_time_sec = battery_joules / total_power_w
    total_frames = int(total_flight_time_sec / dt_real)

    # Internal State Tracking
    state = {
        "pos": np.copy(BASE_POS),
        "batt": battery_joules,
        "radius": 50.0,
        "mode": "DEPLOY",
        "mapped": [],
        "angle": np.pi,
        "elapsed_time": 0.0,
        "total_dist_m": 0.0
    }

    fig, ax = plt.subplots(figsize=(10, 10)) # Slightly larger figure for 3000m map

    def update(frame):
        if state["batt"] <= 0 or state["mode"] == "DONE": return

        # Update Environment & Consumption
        state["radius"] += fire_growth_rate * dt_real
        state["elapsed_time"] += dt_real
        state["batt"] -= total_power_w * dt_real
        
        # Safety & Return Logic
        dist_to_nest = np.linalg.norm(state["pos"] - BASE_POS)
        energy_to_return = (dist_to_nest * 1.4 / speed_ms) * total_power_w
        
        if state["batt"] <= energy_to_return and state["mode"] != "RETURNING":
            state["mode"] = "RETURNING"

        step_dist = speed_ms * dt_real
        state["total_dist_m"] += step_dist
        
        # Navigation (Avoidance + Pathing)
        if state["mode"] == "DEPLOY":
            target = FIRE_CENTER + np.array([-state["radius"], 0])
            dir_vec = target - state["pos"]
            d = np.linalg.norm(dir_vec)
            if d < step_dist: state["mode"] = "MAPPING"
            else: state["pos"] += (dir_vec / d) * step_dist
                
        elif state["mode"] == "MAPPING":
            angular_vel = speed_ms / state["radius"]
            state["angle"] += angular_vel * dt_real
            state["pos"] = FIRE_CENTER + np.array([
                state["radius"] * np.cos(state["angle"]),
                state["radius"] * np.sin(state["angle"])
            ])
            state["mapped"].append(np.copy(state["pos"]))
            
        elif state["mode"] == "RETURNING":
            vec_to_base = BASE_POS - state["pos"]
            d_to_base = np.linalg.norm(vec_to_base)
            avoid_radius = state["radius"] * 1.2
            d_to_fire = np.linalg.norm(state["pos"] - FIRE_CENTER)
            
            if d_to_fire < avoid_radius:
                perp_vec = state["pos"] - FIRE_CENTER
                target = FIRE_CENTER + (perp_vec / (np.linalg.norm(perp_vec) + 0.1)) * avoid_radius
                dir_vec = target - state["pos"]
                state["pos"] += (dir_vec / (np.linalg.norm(dir_vec) + 0.1)) * step_dist
            else:
                state["pos"] += (vec_to_base / d_to_base) * step_dist
                if d_to_base < step_dist: state["mode"] = "DONE"

        # Rendering & Full HUD
        ax.clear()
        ax.set_xlim(0, MAP_SIZE); ax.set_ylim(0, MAP_SIZE)
        batt_pct = (state["batt"] / max_battery) * 100
        total_km = state["total_dist_m"] / 1000.0
        
        ax.set_title(f"3000m Map Mission | Time: {state['elapsed_time']/60:.1f} min")
        
        # Draw Fire and Path
        ax.add_patch(plt.Circle(FIRE_CENTER, state["radius"], color='red', alpha=0.3, label="Fire Line"))
        if state["mapped"]:
            mx, my = zip(*state["mapped"])
            ax.plot(mx, my, color='green', linewidth=1.2, alpha=0.6, label="Mapped Path")
        
        # Drone and Home
        ax.add_patch(plt.Circle(state["pos"], 80, color='yellow', alpha=0.5, label="Neuromorphic FOV"))
        ax.plot(state["pos"][0], state["pos"][1], 'k^', markersize=7)
        ax.plot(BASE_POS[0], BASE_POS[1], 'ks', markersize=12, label="Base Nest")
        
        # FULL TELEMETRY HUD
        hud_text = (
            f"--- HUD STATUS ---\n"
            f"MODE: {state['mode']}\n"
            f"BATTERY: {batt_pct:.1f}%\n"
            f"POWER DRAW: {total_power_w:.1f} W\n"
            f"DIST TO NEST: {dist_to_nest/1000:.2f} km\n"
            f"TOTAL TRAVEL: {total_km:.2f} km"
        )
        ax.text(100, 2900, hud_text, fontsize=11, family='monospace',
                bbox=dict(facecolor='white', alpha=0.85), verticalalignment='top')
        ax.legend(loc="upper right")

    ani = FuncAnimation(fig, update, frames=total_frames, interval=1000/fps)
    ani.save(save_path, writer=PillowWriter(fps=fps))
    plt.close()
    print(f"Simulation saved to {save_path}. Map: {MAP_SIZE}m, Frame Rate: {fps} FPS.")



### Mission run with max speed

In [23]:
run_comprehensive_mission(
    battery_wh=526.4, 
    hover_p=717.0, 
    lift_gain=0.20, 
    speed_ms=20.0, 
    sensor_p=0.01,
    fire_growth_rate=0.2, # Meters per real-world second
    real_to_sim_ratio=60, # 1s sim = 300s (5 min) real
    save_path="SNN_Mission_20ms.gif"
)

Simulation saved to SNN_Mission_20ms.gif. Map: 3000m, Frame Rate: 30 FPS.


### Mission run standard drone

In [22]:
run_comprehensive_mission(
    battery_wh=526.4, 
    hover_p=717.0, 
    lift_gain=0.20, 
    speed_ms=12.0, 
    sensor_p=28,
    fire_growth_rate=0.2, # Meters per real-world second
    real_to_sim_ratio=120, # 1s sim = 300s (5 min) real
    save_path="Standard_mission_12ms.gif"
)

Simulation saved to Standard_mission_12ms.gif. Map: 3000m, Frame Rate: 30 FPS.
